In [ ]:
# User inputs folder with images of same flake, same place, different contrast settings
# Then user creates a mask for the flake(s)
# The contrast of the flake points with the background is calculated and plotted for each image

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np
import sys

from scripts.plotting_functions import create_heatmap_plot, plot_gaussians
from scripts.preprocessor_functions import get_contrasts_from_dir
from scripts.annotation_class import watershed_annotator
from scripts.fitting_functions import fit_set
from scripts.postprocessing_functions import format_components

import time
import json

## Setting some constants

| Parameter             | Description                                                                                                                                    |
| --------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------- |
| `FLAKE_DIRECTORY`     | The directory path of the saved images; the images need to be in `.jpg`, `.png` , or `.tif` format.                                                       |
| `MASK_SAVE_DIRECTORY` | The directory path where the semantic masks are saved.                                                       |
| `ANNOTATE`            | If the images are already annotated or not, if you want to annotate images and save the results to `MASK_SAVE_DIRECTORY` toggle this to `TRUE` |
| `USE_FLATFIELD`       | Whether to use a flatfield image or not (It is highly recommended to use a flatfield image)                                                                                                        |
| `FLATFIELD_PATH`      | The Path to the flatfield image                                                                                                                |
| `USED_CHANNELS `      | The channels used to fit the GM-Model, if, for example, your Blue Channel is Weak you may want to only use `RG`                                |
| `AXIS_NAMES`          | The names used when plotting for the axes                                                                                                      |

As the Supplied Datasets already have their flatfield removed, the default for `USE_FLATFIELD` is set to `False`.  
If you want to use a flatfield image, you need to set `USE_FLATFIELD` to `True` and set the path to the flatfield image in `FLATFIELD_PATH`.


In [ ]:
IMAGE_DIRECTORY = r"../Datasets/GMMDetectorDatasets/usable_TIS_camera/train_images"
MASK_SAVE_DIRECTORY = r"..Datasets/GMMDetectorDatasets/usable_TIS_camera/train_masks"
FLATFIELD_PATH = r"../Datasets/GMMDetectorDatasets/usable_TIS_camera/flatfield_NDTiffStack.tif"

CREATE_NEW_MASK = True

USED_CHANNELS = "BGR"
AXIS_NAMES = ["Blue Contrast", "Green Contrast", "Red Contrast"]

## Checking if the given Parameters are valid


In [ ]:
assert os.path.exists(IMAGE_DIRECTORY), "Flake directory does not exist"
assert os.path.exists(FLATFIELD_PATH), "Flatfield image does not exist"

assert len(USED_CHANNELS) in [2, 3], "We need 2 or 3 channels for the GMM"

os.makedirs(MASK_SAVE_DIRECTORY, exist_ok=True)

## Annotating the Images in the Folder

The annotator uses the watershed algorithm to discern the foreground from the background.  
It is only necessary to define these two classes as the thickness is inferred later by the clustering.

A red outline is shown between the background and foreground indicating the mask boundry.

The Masks are saved as `.png` files in the `MASK_SAVE_DIRECTORY` Folder with the same name as the image.

### Controls

|            Keys             | Description                                                  |
| :-------------------------: | ------------------------------------------------------------ |
| <kbd>A</kbd> / <kbd>D</kbd> | Previous / Next Image                                        |
|        <kbd>S</kbd>         | Save the current annotion, very important after every image! |
|        <kbd>C</kbd>         | Delete the current annotations                               |
|    <kbd>Left MBT </kbd>     | Set a marker for foreground                                  |
|    <kbd>Right MBT </kbd>    | Set a marker for background                                  |
|       <kbd>ESC</kbd>        | Exit the program                                             |


In [ ]:
if CREATE_NEW_MASK:
    annotator = watershed_annotator(
        image_directory=IMAGE_DIRECTORY,
        mask_directory=MASK_SAVE_DIRECTORY,
    )
    annotator.run()

## Calculate contrast graphs
Now it calculates the background color from the flatfield, and uses the mask(s) to calculate flake colors. Then the contrast between the background and flake colors are plotted on graphs for each image.

In [ ]:
USE_ONE_MASK = True

mask_files = [f for f in os.listdir(MASK_SAVE_DIRECTORY) if f.endswith('.tif', '.png', '.jpg')]
if not mask_files:
    print("No mask files found in the specified directory.")
    sys.exit(1)

image_files = [f for f in os.listdir(IMAGE_DIRECTORY) if f.endswith('.tif', '.png', '.jpg')]
if not image_files:
    print("No image files found in the specified directory.")
    sys.exit(1)

if USE_ONE_MASK:
    mask_name = mask_files[0]
    mask_path = os.path.join(MASK_SAVE_DIRECTORY, mask_name)
    print(f"Using mask: {mask_name}")


datapoints_contrast = get_contrasts_from_dir(
    image_directory=IMAGE_DIRECTORY,
    mask_directory=MASK_SAVE_DIRECTORY,
    use_flatfield=True,
    flatfield_path=FLATFIELD_PATH,
)

